# Exercise 13.2: Cross-Entropy Loss and Perplexity

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuhung16au/Maths-for-IT-Notebooks/blob/main/Chapter-13/02-cross-entropy-loss.ipynb)

**Book Reference:** Chapter 13 — Information Theory, Entropy and Loss Functions (`chapters/entropy-and-loss-functions.tex`, Sections 2–3)  
**Estimated Time:** 15–20 min

### Learning Objectives
- Implement cross-entropy $H(P, Q) = -\sum p_i \log(q_i)$ in both bits and nats.
- Verify the identity $H(P, P) = H(P)$.
- Compute next-token negative log-likelihood loss $L = -\ln Q_y$ and perplexity $\exp(\bar L)$.


In [ ]:
import math
import numpy as np

### Mathematical Context

Given true distribution $P$ and predicted distribution $Q$, **cross-entropy** is:
$$H(P, Q) = -\sum_{i=1}^k p_i \log Q_i.$$
When $Q = P$, cross-entropy reduces to Shannon entropy: $H(P, P) = H(P)$.

In LLM next-token prediction and supervised classification, $P$ is **one-hot** ($p_y = 1$ for the true token $y$, $0$ elsewhere), so natural-log cross-entropy simplifies to:
$$L = -\ln Q_y.$$
In the chapter's worked example (true token `"cat"`):
- Model A assigns $Q_A(\text{"cat"}) = 0.8 \implies L_A = -\ln(0.8) \approx 0.223$.
- Model B assigns $Q_B(\text{"cat"}) = 0.1 \implies L_B = -\ln(0.1) \approx 2.303$.

Over a sequence of $N$ tokens with average loss $\bar L$, **perplexity** is $\exp(\bar L)$.

In [ ]:
def cross_entropy(p: np.ndarray, q: np.ndarray, base: float | None = None) -> float:
    """Compute cross-entropy H(P, Q) = -sum_{p_i > 0} p_i * log(q_i).

    Args:
        p: 1D true probability distribution array.
        q: 1D predicted probability distribution array (same shape as p).
        base: Logarithm base; if None (default), uses natural logarithm (nats).

    Returns:
        Cross-entropy H(P, Q) as a float.

    Raises:
        ValueError: If distributions are invalid or q_i <= 0 where p_i > 0.
    """
    # TODO: Validate p and q, check q[p > 0] > 0, and compute -sum(p[mask] * log(q[mask])).
    raise NotImplementedError("Complete this TODO")


def nll_loss_and_perplexity(q_true_probs: np.ndarray) -> tuple[float, float]:
    """Compute mean negative log-likelihood loss and perplexity for one-hot targets.

    Formulas:
        L_bar = -(1 / N) * sum(ln(q_y_i))
        perplexity = exp(L_bar)

    Args:
        q_true_probs: 1D NumPy array of model probabilities assigned to the true tokens.

    Returns:
        Tuple (mean_nll, perplexity).
    """
    # TODO: Validate all probabilities in (0, 1], compute mean(-np.log(q_true_probs)) and math.exp(mean_nll).
    raise NotImplementedError("Complete this TODO")

<details>
<summary>Hint 1: Support of P vs Q</summary>

Only indices where `p > 0` contribute to the sum. If `q_i == 0` at any index where `p_i > 0`, cross-entropy diverges to $+\infty$, so raise `ValueError`.

</details>

<details>
<summary>Hint 2: Perplexity</summary>

Perplexity is simply `math.exp(mean_nll)` where `mean_nll` is computed with natural logarithms (`np.log`). For example, if a model always assigns `1/V` to the true token, perplexity equals `V`.

</details>

In [ ]:
# Self-check assertions
p = np.array([0.5, 0.25, 0.25])
# H(P, P) == H(P)
assert abs(cross_entropy(p, p, base=2.0) - 1.5) < 1e-12

# Worked example: Q_A("cat") = 0.8 vs Q_B("cat") = 0.1
p_onehot = np.array([1.0, 0.0, 0.0])
q_a = np.array([0.8, 0.15, 0.05])
q_b = np.array([0.1, 0.60, 0.30])
loss_a = cross_entropy(p_onehot, q_a)
loss_b = cross_entropy(p_onehot, q_b)
assert abs(loss_a - (-math.log(0.8))) < 1e-12
assert abs(loss_b - (-math.log(0.1))) < 1e-12

mean_nll, ppl = nll_loss_and_perplexity(np.array([0.25, 0.25, 0.25]))
assert abs(ppl - 4.0) < 1e-9, f"Uniform over 4 choices should have perplexity 4, got {ppl}"

print(f"Model A loss: {loss_a:.3f} | Model B loss: {loss_b:.3f}")
print(f"Uniform 0.25 perplexity: {ppl:.2f}")
print("All assertions passed!")

### Think Deeper

1. Why does assigning $Q_y = 0.001$ to the true next token incur a much larger penalty ($-\ln(0.001) \approx 6.91$) than assigning $Q_y = 0.5$ ($-\ln(0.5) \approx 0.693$)?
2. If an LLM has perplexity $12$ on a held-out corpus, how can you interpret that number intuitively?